# Mancino dataset: data audit and cleaning

This notebook starts the data-preparation work before loading the prototype data into MySQL. It examines the four currently proposed source files: transactions, transaction line items, products, and stores. The file list is provisional and can be adjusted after reviewing the findings.

Source CSVs are read-only. The notebook reports potential issues and prepares in-memory copies for review; it does not overwrite source files or write cleaned output files. We will agree on cleaning rules from the audit results rather than silently deleting or changing records.

## 1. Setup and load the proposed source files

Update `DATA_DIR` if the dataset version folder changes. IDs are loaded as text where appropriate to preserve identifier formatting.

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('..') / 'Data' / 'v1_2025-11-01'
CSV_FILES = {
    'transactions': 'transactions.csv',
    'line_items': 'transaction_line_items.csv',
    'products': 'products.csv',
    'stores': 'stores.csv',
}
ID_DTYPES = {
    'transactions': {'transaction_id': 'string', 'customer_id': 'string'},
    'line_items': {'transaction_id': 'string', 'product_id': 'string'},
    'products': {'product_id': 'string'},
    'stores': {},
}
missing_files = [f for f in CSV_FILES.values() if not (DATA_DIR / f).is_file()]
if missing_files:
    raise FileNotFoundError(f'Missing files in {DATA_DIR}: {missing_files}')
raw = {name: pd.read_csv(DATA_DIR / filename, dtype=ID_DTYPES[name]) for name, filename in CSV_FILES.items()}
{name: frame.shape for name, frame in raw.items()}

{'transactions': (167823, 10),
 'line_items': (504000, 8),
 'products': (1500, 8),
 'stores': (5, 9)}

## 2. Initial inspection

Review table sizes, column types, and sample rows before deciding what to clean.

In [2]:
for name, frame in raw.items():
    print(f'\n### {name}: {len(frame):,} rows × {len(frame.columns)} columns')
    display(frame.head())
    display(frame.dtypes.rename('dtype').to_frame())


### transactions: 167,823 rows × 10 columns


,transaction_id,customer_id,store_id,txn_ts,subtotal,tax,total,num_items,payment_type,day_name
0,TX-20250102-000001,C004707,1,2025-01-02 19:25:00,36.12,3.01,36.96,4,DEBIT,Thursday
1,TX-20250102-000002,C004828,1,2025-01-02 19:47:00,12.34,1.01,12.37,3,DEBIT,Thursday
2,TX-20250102-000003,C003241,1,2025-01-02 18:11:00,29.40,2.61,32.01,5,CASH,Thursday
3,TX-20250102-000004,C004413,1,2025-01-02 14:33:00,36.08,3.13,38.40,4,CASH,Thursday
4,TX-20250102-000005,C001677,1,2025-01-02 11:26:00,38.38,3.28,40.19,6,CASH,Thursday


,dtype
transaction_id,string[python]
customer_id,string[python]
store_id,int64
txn_ts,object
subtotal,float64
tax,float64
total,float64
num_items,int64
payment_type,object
day_name,object



### line_items: 504,000 rows × 8 columns


,transaction_id,line_no,product_id,qty,unit_price,line_subtotal,line_discount,line_total
0,TX-20250102-000001,1,P01160,2,10.83,21.66,2.17,19.49
1,TX-20250102-000001,2,P00165,1,7.04,7.04,0.00,7.04
2,TX-20250102-000001,3,P00758,1,7.42,7.42,0.00,7.42
3,TX-20250102-000002,1,P00863,2,4.89,9.78,0.98,8.80
4,TX-20250102-000002,2,P00697,1,2.56,2.56,0.00,2.56


,dtype
transaction_id,string[python]
line_no,int64
product_id,string[python]
qty,int64
unit_price,float64
line_subtotal,float64
line_discount,float64
line_total,float64


... (2 outputs left out)

,dtype
product_id,string[python]
product_name,object
brand,object
category,object
subcategory,object
unit_size,object
price,float64
unit_cost,float64



### stores: 5 rows × 9 columns


,store_id,store_name,neighborhood,address,city,state,zip,latitude,longitude
0,1,Mancino Market - Midtown,Midtown,226 E 57th St,New York,NY,10022,40.759710,-73.96616
1,2,Mancino Market - NoHo,Noho,352 Bowery,New York,NY,10012,40.726600,-73.99170
2,3,Mancino Market - Chelsea,Chelsea,75 9th Ave,New York,NY,10011,40.742352,-74.00621
3,4,Mancino Market - Greenwich Village,Greenwich Village,378 6th Ave,New York,NY,10011,40.733037,-73.99976
4,5,Mancino Market - Nolita,Nolita,202 Mott St,New York,NY,10012,40.721390,-73.99545


,dtype
store_id,int64
store_name,object
neighborhood,object
address,object
city,object
state,object
zip,int64
latitude,float64
longitude,float64


## 3. Missing values and duplicate rows

Diagnostic only: review flagged values before removing rows or filling fields.

In [3]:
for name, frame in raw.items():
    print(f'\n### {name}')
    print('Rows:', len(frame), '| duplicate full rows:', int(frame.duplicated().sum()))
    display(frame.isna().sum().rename('missing_count').to_frame().query('missing_count > 0'))


### transactions
Rows: 167823 | duplicate full rows: 0


,missing_count



### line_items
Rows: 504000 | duplicate full rows: 0


,missing_count



### products
Rows: 1500 | duplicate full rows: 0


,missing_count



### stores
Rows: 5 | duplicate full rows: 0


,missing_count


## 4. Key uniqueness and table relationships

Check expected business keys and references before defining database constraints.

In [4]:
key_specs = {
    'transactions': ['transaction_id'],
    'line_items': ['transaction_id', 'line_no'],
    'products': ['product_id'],
    'stores': ['store_id'],
}
for name, keys in key_specs.items():
    frame = raw[name]
    print(f'{name}: null key rows = {int(frame[keys].isna().any(axis=1).sum()):,}; duplicate keys = {int(frame.duplicated(keys).sum()):,}')

tx = raw['transactions']
li = raw['line_items']
products = raw['products']
stores = raw['stores']
print('Line items without a transaction:', int((~li['transaction_id'].isin(tx['transaction_id'])).sum()))
print('Line items without a product:', int((~li['product_id'].isin(products['product_id'])).sum()))
known_store_ids = set(stores['store_id'].astype('string'))
print('Transactions without a store:', int((~tx['store_id'].astype('string').isin(known_store_ids)).sum()))

transactions: null key rows = 0; duplicate keys = 0
line_items: null key rows = 0; duplicate keys = 0
products: null key rows = 0; duplicate keys = 0
stores: null key rows = 0; duplicate keys = 0
Line items without a transaction: 0
Line items without a product: 0
Transactions without a store: 0


## 5. Field and calculation checks

These checks flag suspicious values but do not assume every mismatch is an error. Confirm field meanings before using `line_total` for revenue and `unit_cost × qty` for estimated cost of goods sold.

In [5]:
items = li.copy()
numeric_cols = ['line_no', 'qty', 'unit_price', 'line_subtotal', 'line_discount', 'line_total']
for col in numeric_cols:
    items[col] = pd.to_numeric(items[col], errors='coerce')
print('Non-positive quantities:', int((items['qty'] <= 0).sum()))
print('Negative unit prices:', int((items['unit_price'] < 0).sum()))
print('Negative discounts:', int((items['line_discount'] < 0).sum()))
tolerance = 0.02
subtotal_mismatch = (items['qty'] * items['unit_price'] - items['line_subtotal']).abs() > tolerance
net_mismatch = (items['line_subtotal'] - items['line_discount'] - items['line_total']).abs() > tolerance
print('qty × unit_price differs from line_subtotal:', int(subtotal_mismatch.sum()))
print('line_subtotal - discount differs from line_total:', int(net_mismatch.sum()))
display(items.loc[subtotal_mismatch | net_mismatch].head(20))

Non-positive quantities: 0
Negative unit prices: 0
Negative discounts: 0
qty × unit_price differs from line_subtotal: 0
line_subtotal - discount differs from line_total: 0


,transaction_id,line_no,product_id,qty,unit_price,line_subtotal,line_discount,line_total


## 6. Date, category, and transaction reconciliation checks

Compare transaction headers with line-item amounts to understand which fields should drive the dashboard. Header totals may include tax or other adjustments, so differences are diagnostic, not automatic failures.

In [6]:
tx_check = tx.copy()
tx_check['txn_ts_parsed'] = pd.to_datetime(tx_check['txn_ts'], errors='coerce')
print('Unparseable timestamps:', int(tx_check['txn_ts_parsed'].isna().sum()))
print('Date range:', tx_check['txn_ts_parsed'].min(), 'to', tx_check['txn_ts_parsed'].max())
print('Payment types:', sorted(tx['payment_type'].dropna().astype(str).str.strip().unique()))
print('Product categories:', sorted(products['category'].dropna().astype(str).str.strip().unique()))
line_sums = items.groupby('transaction_id').agg(sum_line_subtotal=('line_subtotal', 'sum'), sum_line_total=('line_total', 'sum')).reset_index()
reconciliation = tx[['transaction_id', 'subtotal', 'tax', 'total']].merge(line_sums, on='transaction_id', how='outer', indicator=True)
for col in ['subtotal', 'tax', 'total', 'sum_line_subtotal', 'sum_line_total']:
    reconciliation[col] = pd.to_numeric(reconciliation[col], errors='coerce')
reconciliation['header_subtotal_minus_lines'] = reconciliation['subtotal'] - reconciliation['sum_line_subtotal']
reconciliation['header_total_minus_net_lines'] = reconciliation['total'] - reconciliation['sum_line_total']
display(reconciliation['_merge'].value_counts().rename('transaction_count').to_frame())
display(reconciliation[['header_subtotal_minus_lines', 'header_total_minus_net_lines']].describe(percentiles=[.5, .9, .99]))
display(reconciliation.loc[reconciliation['_merge'] != 'both'].head(20))

Unparseable timestamps: 0
Date range: 2025-01-01 08:02:00 to 2025-11-01 20:57:00
Payment types: ['CASH', 'CREDIT', 'DEBIT', 'WALLET']
Product categories: ['Alcohol', 'Bakery', 'Baking', 'Beverages', 'Breakfast & Cereal', 'Candy', 'Condiments/Sauces/Spices', 'Dairy & Eggs', 'Deli', 'Frozen Foods', 'Grains and Pasta', 'Meat and Seafood', 'Pet Food', 'Prepared Meals', 'Snacks/Cookies/Chips']


,transaction_count
_merge,
both,167823
left_only,0
right_only,0


,header_subtotal_minus_lines,header_total_minus_net_lines
count,1.678230e+05,167823.000000
mean,2.285502e-17,2.368774
std,2.183725e-15,1.400039
min,-1.421085e-14,0.080000
50%,0.000000e+00,2.130000
90%,0.000000e+00,4.270000
99%,7.105427e-15,6.600000
max,2.842171e-14,14.960000


,transaction_id,subtotal,tax,total,sum_line_subtotal,sum_line_total,_merge,header_subtotal_minus_lines,header_total_minus_net_lines


## 7. Cleaning decisions

After reviewing the outputs, record each accepted rule here before applying it. Potential rules may cover parsing dates, normalizing labels, resolving duplicate business keys, and handling invalid relationships. Do not drop records only because a diagnostic flags them.

Once the rules are agreed, add a final section that creates cleaned in-memory DataFrames and rechecks keys and relationships. Keep source files unchanged; cleaned outputs and the MySQL ingestion script will be handled separately.